# 1. Indexing

## Installing dependencies

In [5]:
!pip install "weaviate-client>=4.0.0"
!pip install tqdm
!pip install tran
!pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


## Importing dependencies

In [7]:
import socket

import weaviate
from sentence_transformers import SentenceTransformer
from weaviate.classes.config import Property, DataType
from weaviate.classes.query import MetadataQuery

/home/dltec/env/3.12/lib/python3.12/site-packages/authlib/integrations/httpx_client/assertion_client.py:5: AuthlibDeprecationWarning: The httpx module is deprecated; please use httpx2 instead.
  from ._compat import httpx2
/home/dltec/env/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Starting Database

If database was started it will be used else it will be created and started.

In [8]:
HTTP_PORT = 8079
GRPC_PORT = 50050


def already_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", port)) == 0


if already_in_use(HTTP_PORT) or already_in_use(GRPC_PORT):
    print("Weaviate is running. Connected by connect_to_local...")
    client = weaviate.connect_to_local(
        port=HTTP_PORT,
        grpc_port=GRPC_PORT
    )
else:
    print("Starting Weaviate Embedded...")
    client = weaviate.connect_to_embedded(
        persistence_data_path="../../resources/weaviate/database",
        environment_variables={
            "ENABLE_MODULES": "",
            "LOG_LEVEL": "error"
        }
    )

Starting Weaviate Embedded...


/home/dltec/env/3.12/lib/python3.12/site-packages/weaviate/embedded.py:148: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  binary_tar.extract("weaviate", path=Path(self.options.binary_path))
/home/dltec/env/3.12/lib/python3.12/site-packages/weaviate/warnings.py:93: DeprecationWarning: Dep005: You are using weaviate-client version 4.16.7. The latest version is 4.23.1.
            Consider upgrading to the latest version. See https://weaviate.io/developers/weaviate/client-libraries/python for details.
  warnings.warn(


## Creating collection

In [9]:
import weaviate.classes as wvc

try:
    collection = client.collections.get("Documents")
    client.collections.delete("Documents")
    print("Collection exists, cleaning...")
except weaviate.exceptions.CollectionNotFoundError:
    print("Collection does not exists, skipping clean...")

collection = client.collections.create(
    name="Documents",
    properties=[Property(name="content", data_type=DataType.TEXT)],
    vector_config=wvc.config.Configure.Vectors.self_provided(),
)
print("Collection created!")

Collection exists, cleaning...
Collection created!


## Generating embeddings

In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [
    """
    The New York subway hums with a rhythm that never really stops.
Steel trains roar through tunnels carved deep beneath the city’s pulse.
Straphangers from every walk of life share those crowded platforms.
A mosaic of sounds—buskers, announcements, screeching rails—fills the air.
And no matter the hour, it always feels like the city is moving right beside you.
    """,
    """
    Cats move with a quiet elegance that feels almost magical.
Their curious eyes catch every flicker of light.
A soft purr can turn any bad moment into calm.
They nap like it’s an art form perfected over centuries.
And despite their independence, they always find a way to your heart.
    """,
    """Weaviate is an AI‑native vector database designed to store and search data using embeddings. 
It supports hybrid search, combining vector similarity with keyword queries for more accurate results.
Its modular architecture allows integration with models like OpenAI, Cohere, and Hugging Face.
Weaviate also provides scalable clustering, making it suitable for large‑scale semantic search applications.
Developers use it to build recommendation systems, chatbots, and knowledge graphs powered by machine learning."""
]
embeddings = model.encode(texts)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1701.44it/s]


In [11]:
print(f"Items: {len(embeddings)}")
print(f"Dimensions: {len(embeddings[0])}")

Items: 3
Dimensions: 384


## Saving Embeddings

In [12]:
docs = list()
for text, emb in zip(texts, embeddings):
    docs.append(wvc.data.DataObject(
        properties={
            "content": text
        },
        vector=emb
    ))

collection.data.insert_many(docs)

BatchObjectReturn(_all_responses=[UUID('622028c2-75ed-4742-9449-d0a9668e8120'), UUID('32bb1f49-a848-4c62-b0f7-1eb82cffc062'), UUID('21f22df7-3598-453d-a27c-586782181a6a')], elapsed_seconds=0.028310537338256836, errors={}, uuids={0: UUID('622028c2-75ed-4742-9449-d0a9668e8120'), 1: UUID('32bb1f49-a848-4c62-b0f7-1eb82cffc062'), 2: UUID('21f22df7-3598-453d-a27c-586782181a6a')}, has_errors=False)

# 2. Retrieval

In [13]:
#query = "how weaviate works?"
query = "how cats moves?"
query_vector = model.encode(query)

response = collection.query.hybrid(
    query=query,
    alpha=0.80,
    vector=query_vector,
    limit=1,
    return_metadata=MetadataQuery(score=True, explain_score=True),
)

In [14]:
result = response.objects[0]

print("Score:", result.metadata.score)
print("Texto:", result.properties["content"])

Score: 1.0
Texto: 
    Cats move with a quiet elegance that feels almost magical.
Their curious eyes catch every flicker of light.
A soft purr can turn any bad moment into calm.
They nap like it’s an art form perfected over centuries.
And despite their independence, they always find a way to your heart.
    


In [15]:
client.close()

{"build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","error":"cannot find peer","level":"error","msg":"transferring leadership","time":"2026-10-07T23:14:42-03:00"}
